# Pure water/steam condensation inside tubes (v0.6.2)

Public `BareTubeHeatExchanger.simulate()` examples for saturated, wet and superheated steam, complete condensation with subcooling, and a low-mass-flux industrial bundle. The calculation uses pressure/enthalpy equilibrium and vapor quality; it does not use wet-gas `W`, dew point, Lewis number or wet-surface fraction.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / 'core').is_dir()), None)
if root is None:
    raise RuntimeError('Repository root was not found.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from core.geometry.bundle import TubeBundle
from core.geometry.tube import BareTube
from core.models.bare_tube import BareTubeHeatExchanger
from core.models.simulation import HXSideInput
from core.phase_change.steam_condensation import SteamTubeOrientation
from core.properties.common import FluidTransportProperties
from core.properties.fluids import ConstantPropertyProvider
from core.properties.water import IAPWS97WaterSteamProvider

P_STEAM = 1.0e6
ORIENTATION = SteamTubeOrientation.VERTICAL_DOWNWARD
steam = IAPWS97WaterSteamProvider()
air = ConstantPropertyProvider(
    FluidTransportProperties(rho=1.2, mu=1.8e-5, k=0.026, cp=1005.0)
)

def exchanger(n_rows=10, n_tubes_per_row=10):
    return BareTubeHeatExchanger(TubeBundle(
        tube=BareTube(D_i=0.020, D_o=0.024, length_total=4.0,
                      length_effective=4.0, wall_k=16.0),
        n_rows=n_rows, n_tubes_per_row=n_tubes_per_row,
        pitch_transverse=0.040, pitch_longitudinal=0.040,
        layout='inline', n_passes_tube=1, flow_arrangement='crossflow',
    ))

def run_case(name, state, m_dot, *, n_rows=10, n_tubes_per_row=10, outside_m_dot=30.0):
    hx = exchanger(n_rows, n_tubes_per_row)
    inside = HXSideInput(provider=steam, m_dot=m_dot, p=P_STEAM,
                         steam_tube_orientation=ORIENTATION, **state)
    outside = HXSideInput(provider=air, m_dot=outside_m_dot,
                          T_in=300.0, p=101325.0)
    return name, state, hx.simulate(inside, outside)

## Five representative Simulation cases

Case D traverses all three zones. Case E deliberately uses 400 parallel tubes so total-water mass flux is only a few kg/(m²·s); Shah's gravity/film contribution prevents the condensation HTC from collapsing.

In [2]:
runs = [
    run_case('A saturated vapor -> partial', {'quality_in': 1.0}, 1.0),
    run_case('B wet inlet -> lower quality', {'quality_in': 0.8}, 1.0),
    run_case('C superheated -> condensation', {'T_in': 520.0}, 1.0),
    run_case('D superheated -> complete -> subcooled', {'T_in': 520.0}, 0.25),
    run_case('E low-G industrial bundle', {'quality_in': 1.0}, 0.5,
             n_rows=20, n_tubes_per_row=20, outside_m_dot=20.0),
]

rows = []
for name, state_spec, result in runs:
    pc = result.inside_phase_change
    rows.append({
        'case': name, 'input_state': repr(state_spec),
        'phase_in': pc.phase_in.value, 'phase_out': pc.phase_out.value,
        'T_in_K': pc.T_in, 'T_out_K': pc.T_out, 'Tsat_K': pc.Tsat,
        'h_in_kJkg': pc.h_in / 1e3, 'h_out_kJkg': pc.h_out / 1e3,
        'quality_in': pc.quality_in, 'quality_out': pc.quality_out,
        'Q_desuperheat_kW': pc.Q_desuperheat / 1e3,
        'Q_condensation_kW': pc.Q_condensation / 1e3,
        'Q_subcooling_kW': pc.Q_subcooling / 1e3,
        'Q_total_kW': pc.Q_total / 1e3,
        'A_desuperheat_m2': pc.A_desuperheat,
        'A_condensation_m2': pc.A_condensation,
        'A_subcooling_m2': pc.A_subcooling, 'A_total_m2': pc.A_total,
        'fraction_desuperheat': pc.zone_fraction_desuperheat,
        'fraction_condensation': pc.zone_fraction_condensation,
        'fraction_subcooling': pc.zone_fraction_subcooling,
        'zone_alpha_desuperheat_W_m2K': pc.zone_alpha_desuperheat,
        'zone_alpha_condensation_W_m2K': pc.zone_alpha_condensation,
        'zone_alpha_subcooling_W_m2K': pc.zone_alpha_subcooling,
        'zone_UA_desuperheat_W_K': pc.zone_UA_desuperheat,
        'zone_UA_condensation_W_K': pc.zone_UA_condensation,
        'zone_UA_subcooling_W_K': pc.zone_UA_subcooling,
        'UA_total_W_K': pc.UA_total,
        'top_level_alfa_i_reporting_W_m2K': result.inside_alfa_mean,
        'mass_flux_kg_m2s': pc.solution.mass_flux,
        'two_phase_dp_status': pc.two_phase_pressure_drop_status,
        'warnings': ', '.join(w.code for w in pc.warnings),
    })

summary = pd.DataFrame(rows).set_index('case')
display(summary.round(4))

for _, _, result in runs:
    pc = result.inside_phase_change
    assert abs(pc.Q_total - (pc.Q_desuperheat + pc.Q_condensation + pc.Q_subcooling)) < 1e-6
    assert abs(pc.A_total - (pc.A_desuperheat + pc.A_condensation + pc.A_subcooling)) < 1e-7
    assert abs(pc.UA_total - (pc.zone_UA_desuperheat + pc.zone_UA_condensation + pc.zone_UA_subcooling)) < 1e-7

low_g = runs[-1][2].inside_phase_change
assert low_g.solution.mass_flux < 5.0
assert low_g.zone_alpha_condensation > 1000.0
assert runs[3][2].inside_phase_change.Q_subcooling > 0.0
print('All five public steam-condensation examples passed.')

,input_state,phase_in,phase_out,T_in_K,T_out_K,Tsat_K,h_in_kJkg,h_out_kJkg,quality_in,quality_out,...,zone_alpha_condensation_W_m2K,zone_alpha_subcooling_W_m2K,zone_UA_desuperheat_W_K,zone_UA_condensation_W_K,zone_UA_subcooling_W_K,UA_total_W_K,top_level_alfa_i_reporting_W_m2K,mass_flux_kg_m2s,two_phase_dp_status,warnings
case,,,,,,,,,,,,,,,,,,,,,
A saturated vapor -> partial,{'quality_in': 1.0},saturated_vapor,two_phase,453.0356,453.0356,453.0356,2777.1195,1764.5481,1.0,0.4973,...,12220.7658,NaN,0.0000,7432.0771,0.0000,7432.0771,12220.7658,31.8310,not_supported,"STEAM_CONDENSATION_SHAH_2009_OUTSIDE_RANGE, ST..."
B wet inlet -> lower quality,{'quality_in': 0.8},two_phase,two_phase,453.0356,453.0356,453.0356,2374.2322,1367.0830,0.8,0.3000,...,9777.2655,NaN,0.0000,7387.4027,0.0000,7387.4027,9777.2655,31.8310,not_supported,"STEAM_CONDENSATION_SHAH_2009_OUTSIDE_RANGE, ST..."
C superheated -> condensation,{'T_in': 520.0},superheated_vapor,two_phase,520.0000,453.0356,453.0356,2936.2437,2056.3080,NaN,0.6422,...,13389.2710,NaN,925.5428,5206.5581,0.0000,6132.1009,9421.8454,31.8310,not_supported,"STEAM_CONDENSATION_SHAH_2009_OUTSIDE_RANGE, ST..."
D superheated -> complete -> subcooled,{'T_in': 520.0},superheated_vapor,subcooled_liquid,520.0000,376.1752,453.0356,2936.2437,432.5302,NaN,NaN,...,8213.1468,124.9396,225.8520,3530.2498,791.9215,4548.0233,4002.8042,7.9577,not_supported,"STEAM_CONDENSATION_SHAH_2009_OUTSIDE_RANGE, ST..."
E low-G industrial bundle,{'quality_in': 1.0},saturated_vapor,subcooled_liquid,453.0356,345.4219,453.0356,2777.1195,303.3222,1.0,NaN,...,10260.1816,125.0464,0.0000,8237.8587,3354.9495,11592.8082,5428.4325,3.9789,not_supported,"STEAM_CONDENSATION_SHAH_2009_OUTSIDE_RANGE, ST..."


All five public steam-condensation examples passed.


## Reading the coefficients

`zone_alpha_condensation` is the physical condensation-zone HTC and is the value to validate against Shah (2009). `top_level_alfa_i_reporting_W_m2K` is only the existing area-weighted compatibility/reporting diagnostic. The solver does not use that top-level value to allocate zones or reconstruct physics: authoritative conductance is the displayed sum of the three zone UAs. A condensation zone also reports two-phase pressure drop as unsupported, so no partial single-phase value is presented as a complete tube-side pressure drop.